# 09 — Leitura da avaliação temporal final

A avaliação do HGB congelado, com 19 features, foi concluída pelo autor na revisão
`e0fddc0`. O teste de 20–26/05/2018 foi consumido. Este notebook verifica e interpreta
os artefatos já publicados, sem executar novamente a avaliação ou carregar o modelo.

Preencha o caminho retornado pela execução original. O verificador mantém os checks
de integridade e métricas. Reproduzir a execução exige a revisão histórica e seu
ambiente; uma refatoração não autoriza outro acesso ao holdout.

[Protocolo e recuperação](../../docs/modeling/EVALUATION_PROTOCOL.md#executar-a-avaliação-final).


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from fraud_detection_mlops.config import PROJECT_ROOT
from fraud_detection_mlops.modeling.experiments import (
    final_holdout_evaluation as evaluation,
)

EVALUATION_PATH = Path("CAMINHO_EVALUATION_PATH_RETORNADO")
if not EVALUATION_PATH.is_absolute():
    EVALUATION_PATH = PROJECT_ROOT / EVALUATION_PATH
result = evaluation.verify_evaluation(EVALUATION_PATH)
receipt = json.loads((EVALUATION_PATH / "frozen_candidate.json").read_text())
receipt["policy"]["holdout"], receipt["policy"]["acceptance"]

## Conferir o resultado publicado

A verificação lê predições, métricas e recibos salvos. Não lê Gold, não faz fit,
não cria run MLflow nem consome o teste novamente. O caminho precisa conter os
outputs da avaliação original preservados juntos.


In [ ]:
result

## Métricas e decisão

AP é calculada sobre todas as transações do teste. A precisão @100 corresponde
à média não ponderada das precisões diárias por cliente. Ambos os mínimos
predefinidos devem ser alcançados: AP ≥ 0,50 e precisão diária @100 ≥ 0,45.
`status: success` informa a conclusão da execução; consulte `gate.passed` para
saber se os critérios foram atingidos. A decisão não promove o modelo em produção.


In [ ]:
report = json.loads((EVALUATION_PATH / "report.json").read_text())
comparison = pd.DataFrame(result["comparison"]).T
comparison[
    [
        "rows",
        "fraud_count",
        "fraud_rate",
        "average_precision",
        "roc_auc",
        "daily_customer_precision_at_100",
    ]
]

In [ ]:
display(pd.DataFrame(report["gate"]["criteria"]).T)
report["gate"]["decision"]

In [ ]:
daily = pd.read_csv(EVALUATION_PATH / "daily.csv", parse_dates=["date"])
daily

## Variação diária

Os gráficos ajudam a interpretar estabilidade e dias difíceis. Os critérios do
gate são agregados; não representam limites obrigatórios para cada dia.
O controle tem score constante e desempate determinístico, sem treinamento.
Uma única semana de dados sintéticos não fundamenta superioridade estatística
ou eficácia em pagamentos reais. O ranking usa o dia completo retrospectivo.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
labels = {
    "hist_gradient_boosting": "HGB congelado",
    "constant_score_control": "Score constante",
}
for model_id, group in daily.groupby("model_id"):
    ordered = group.sort_values("date")
    axes[0].plot(ordered.date, ordered.average_precision, marker="o", label=labels[model_id])
    axes[1].plot(ordered.date, ordered.precision_at_100, marker="o", label=labels[model_id])
for axis, title in zip(
    axes, ["Average Precision diária", "Precisão por cliente @100 diária"], strict=True
):
    axis.set_title(title)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.3)
    axis.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
axes[1].set_xlabel("Dia do teste")
fig.suptitle("Avaliação temporal final — dados sintéticos")
fig.tight_layout()
plt.show()

In [ ]:
evaluation.verify_evaluation(EVALUATION_PATH)

## Model Card e próximos passos

A verificação acima reconcilia arquivos e métricas dos scores salvos; não lê o
Gold nem recarrega o modelo. Ela não comprova autenticidade contra quem pode
reescrever simultaneamente os arquivos e seus hashes.

Revise a Model Card gerada e registre a execução real na documentação antes de
seguir ao contrato de inferência e Docker. Se o gate falhar, registre a falha;
não ajuste o modelo ou os critérios com base neste teste.


In [ ]:
display(Markdown((EVALUATION_PATH / "model_card.md").read_text()))